<a href="https://colab.research.google.com/github/4-akshay/4-akshay/blob/main/Hybrid_Quantum_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pip
!pip install pennylane tensorflow matplotlib qiskit


In [ ]:
import pennylane as qml
from pennylane import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt


In [ ]:
# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize and reshape for CNN input
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., None]
x_test = x_test[..., None]

# One-hot encode labels
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)


In [ ]:
def build_feature_extractor(n_features=4):
    inp = layers.Input(shape=(28,28,1))
    x = layers.Conv2D(16, (3,3), activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, (3,3), activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    features = layers.Dense(n_features, activation="tanh")(x)  # range [-1,1]
    return models.Model(inp, features)

feature_extractor = build_feature_extractor()


In [ ]:
n_qubits = 4
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

# Define QNode
@qml.qnode(dev, interface="tf")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(np.pi * inputs[i], wires=i)
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Custom Keras Layer
class QuantumLayer(layers.Layer):
    def __init__(self, n_qubits, n_layers, **kwargs):
        super().__init__(**kwargs)
        init_shape = (n_layers, n_qubits, 3)
        self.weights_q = tf.Variable(tf.random.normal(init_shape), trainable=True)
        self.n_qubits = n_qubits

    def call(self, inputs):
        # Use tf.map_fn to handle dynamic batch size
        outputs = tf.map_fn(
            lambda x: tf.convert_to_tensor(quantum_circuit(x, self.weights_q), dtype=tf.float32),
            inputs,
            fn_output_signature=tf.float32
        )
        return outputs

qlayer = QuantumLayer(n_qubits=n_qubits, n_layers=n_layers)


In [ ]:
inp = layers.Input(shape=(28,28,1))
x = feature_extractor(inp)
x = qlayer(x)  # quantum layer
x = layers.Dense(32, activation="relu")(x)
out = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs=inp, outputs=out)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


Cause: mangled names are not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: mangled names are not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: mangled names are not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: mangled names are not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: mangled names are not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


...

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional_2 (Functional)       │ (None, 4)              │        56,324 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ quantum_layer_3 (QuantumLayer)  │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,814 (221.93 KB)

 Trainable params: 56,814 (221.93 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x_train, y_train_cat,
                    batch_size=64,
                    epochs=5,
                    validation_split=0.1)


Epoch 1/5


844/844 ━━━━━━━━━━━━━━━━━━━━ 173s 186ms/step - accuracy: 0.4214 - loss: 1.5890 - val_accuracy: 0.8917 - val_loss: 0.5134
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 197s 181ms/step - accuracy: 0.9128 - loss: 0.3908 - val_accuracy: 0.9722 - val_loss: 0.1489
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 153s 181ms/step - accuracy: 0.9741 - loss: 0.1306 - val_accuracy: 0.9767 - val_loss: 0.1063
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 151s 179ms/step - accuracy: 0.9814 - loss: 0.0807 - val_accuracy: 0.9782 - val_loss: 0.0940
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 153s 181ms/step - accuracy: 0.9847 - loss: 0.0649 - val_accuracy: 0.9828 - val_loss: 0.0856


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat)
print("Test accuracy:", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - accuracy: 0.9783 - loss: 0.0867
Test accuracy: 0.9829000234603882


In [ ]:
!pip install torch
torch.save(model.state_dict(), "quantum_cnn_detector.pth")


NameError: name 'torch' is not defined